# Modeling

Predictive modeling pipeline for forecasting county-level `Early_Delinquency_Rate` one month ahead using a **CatBoost Regressor**.
The dataset is split chronologically to prevent data leakage.

In [ ]:
%reload_ext autoreload
%autoreload 2

import pandas as pd
from catboost import CatBoostRegressor

from climatefinance import modeling, plots, utils

results = []

In [2]:
# Read data
analysis_df = utils.load_analysis_data()
# Fix datetime loading from csv file
analysis_df["month"] = pd.to_datetime(analysis_df["month"])

Loaded 73899 rows from data/analysis/finance_disaster_analysis.csv


## Feature Engineering & Train/Test Split

`build_prediction_dataset` constructs 60 features from the raw panel:
- **Binary event indicators** — `flood_occur`, `tornado_occur`, `thunder_occur`, `hail_occur`
- **Calendar features** — `month_num`, `year`, `quarter`
- **Lagged targets** — 1, 2, 3, 6, 12 month lags of both early and late delinquency rates
- **Lagged disaster features** — 1, 2, 3, 6 month lags of event occurrence, log damage, disaster count, and each type
- **Rolling disaster exposure** — 3- and 6-month rolling sums of event occurrence and log damage

The target is `Early_Delinquency_Rate` shifted forward by 1 month (next-month prediction).

`temporal_split` splits chronologically: train → validation (12 months) → test (12 months).

In [ ]:
model_df = modeling.build_prediction_dataset(
    analysis_df,
    target_col="Early_Delinquency_Rate",
    horizon=1
)

train_df, valid_df, test_df = modeling.temporal_split(
    model_df, date_col="month", test_months=12, valid_months=12
)

## Feature & Target Selection

Define categorical and numeric feature columns, filter to those that exist in the dataset, and split into X/y for train, validation, and test.

In [ ]:
categorical_cols = [
    "fips", "State", "County", "Coastal_County", "Coastline_Region", "month_num", "quarter"
]

feature_cols = [
    # static / slow-moving
    "fips", "State", "County", "Coastal_County", "Coastline_Region",
    "Land_Area", "Distance_To_Coast", "Pop",
    "month_num", "year", "quarter",

    # current disaster conditions
    "event_occur", "n_disasters", "log_total_damage",
    "flood_occur", "tornado_occur", "thunder_occur", "hail_occur",

    # lagged target information
    "Early_Delinquency_Rate_lag1", "Early_Delinquency_Rate_lag2",
    "Early_Delinquency_Rate_lag3", "Early_Delinquency_Rate_lag6",
    "Early_Delinquency_Rate_lag12",

    "Late_Delinquency_Rate_lag1", "Late_Delinquency_Rate_lag2",
    "Late_Delinquency_Rate_lag3", "Late_Delinquency_Rate_lag6",
    "Late_Delinquency_Rate_lag12",

    # lagged disasters
    "event_occur_lag1", "event_occur_lag2", "event_occur_lag3", "event_occur_lag6",
    "log_total_damage_lag1", "log_total_damage_lag2",
    "log_total_damage_lag3", "log_total_damage_lag6",
    "n_disasters_lag1", "n_disasters_lag2", "n_disasters_lag3", "n_disasters_lag6",
    "flood_occur_lag1", "flood_occur_lag2", "flood_occur_lag3", "flood_occur_lag6",
    "tornado_occur_lag1", "tornado_occur_lag2", "tornado_occur_lag3", "tornado_occur_lag6",
    "thunder_occur_lag1", "thunder_occur_lag2", "thunder_occur_lag3", "thunder_occur_lag6",
    "hail_occur_lag1", "hail_occur_lag2", "hail_occur_lag3", "hail_occur_lag6",

    # rolling exposure
    "event_occur_roll3", "event_occur_roll6",
    "log_total_damage_roll3", "log_total_damage_roll6",
]

# Keep only columns that actually exist
feature_cols = [c for c in feature_cols if c in model_df.columns]
categorical_cols = [c for c in categorical_cols if c in feature_cols]

X_train = train_df[feature_cols].copy()
y_train = train_df["target"].copy()

X_valid = valid_df[feature_cols].copy()
y_valid = valid_df["target"].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df["target"].copy()

print("Number of features:", len(feature_cols))

## Naive Baseline

A persistence forecast: predict next month's delinquency rate equals the current month's rate.
This provides the lower bound that the model must beat.

In [ ]:
# Naive baseline: next month ~= current month
naive_pred = test_df["Early_Delinquency_Rate"].values
baseline_metrics = modeling.evaluate_predictions(y_test, naive_pred, name="Naive baseline")
results.append(baseline_metrics)

## CatBoost Training

Categorical columns are filled with `"missing"` for NaN values and cast to string.
CatBoost handles categorical encoding natively via ordered target statistics.

Hyperparameters: 1,500 iterations, learning rate 0.03, depth 6, RMSE loss.
Early stopping selects the best iteration on the validation set.

In [6]:
for col in categorical_cols:
    X_train[col] = X_train[col].fillna("missing").astype(str)
    X_valid[col] = X_valid[col].fillna("missing").astype(str)
    X_test[col] = X_test[col].fillna("missing").astype(str)

cat_model = CatBoostRegressor(
    iterations=1500,
    learning_rate=0.03,
    depth=6,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    verbose=100
)

cat_model.fit(
    X_train,
    y_train,
    cat_features=categorical_cols,
    eval_set=(X_valid, y_valid),
    use_best_model=True
)

0:	learn: 1.2901105	test: 1.1064729	best: 1.1064729 (0)	total: 78.3ms	remaining: 1m 57s
100:	learn: 0.3656997	test: 0.3539819	best: 0.3533419 (95)	total: 929ms	remaining: 12.9s
200:	learn: 0.3318749	test: 0.3220690	best: 0.3220690 (200)	total: 1.83s	remaining: 11.8s
300:	learn: 0.3229933	test: 0.3031110	best: 0.3031110 (300)	total: 2.79s	remaining: 11.1s
400:	learn: 0.3174173	test: 0.2952493	best: 0.2952493 (400)	total: 3.74s	remaining: 10.3s
500:	learn: 0.3130979	test: 0.2935086	best: 0.2931952 (482)	total: 4.74s	remaining: 9.45s
600:	learn: 0.3092816	test: 0.2922078	best: 0.2918497 (581)	total: 5.68s	remaining: 8.49s
700:	learn: 0.3058922	test: 0.2913328	best: 0.2911478 (666)	total: 6.59s	remaining: 7.51s
800:	learn: 0.3034883	test: 0.2908805	best: 0.2908470 (798)	total: 7.5s	remaining: 6.55s
900:	learn: 0.3013850	test: 0.2899780	best: 0.2899358 (897)	total: 8.42s	remaining: 5.59s
1000:	learn: 0.2993081	test: 0.2896162	best: 0.2894472 (967)	total: 9.35s	remaining: 4.66s
1100:	learn: 

CatBoostRegressor(depth=6, eval_metric='RMSE', iterations=1500, learning_rate=0.03, loss_function='RMSE', random_seed=42, verbose=100)

## Evaluation

Compute RMSE, MAE, and R² on both validation and held-out test sets.

In [ ]:
valid_pred = cat_model.predict(X_valid)
test_pred = cat_model.predict(X_test)

valid_metrics = modeling.evaluate_predictions(y_valid, valid_pred, name="CatBoost validation")
test_metrics = modeling.evaluate_predictions(y_test, test_pred, name="CatBoost test")
results.append(valid_metrics)
results.append(test_metrics)

## Feature Importance

Top 20 features by CatBoost's built-in importance metric.
Lagged delinquency rates are expected to dominate (strong autocorrelation), with disaster features providing additional signal.

In [ ]:
plots.plot_model_feature_importance(
    feature_cols, cat_model.get_feature_importance()
);

## Monthly Performance

Average true vs predicted `Early_Delinquency_Rate` by month on the test set.
This shows whether the model tracks the aggregate trend or diverges in specific periods.

In [ ]:
pred_df = test_df[["fips", "County", "month"]].copy()
pred_df["y_true"] = y_test.values
pred_df["y_pred"] = test_pred

monthly_perf = pred_df.groupby("month")[["y_true", "y_pred"]].mean()

plots.plot_model_monthly_performance(
    monthly_perf["y_true"], monthly_perf["y_pred"]
);

In [ ]:
# Save modeling results
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
utils.save_analysis_modeling(results_df, index=False)

## Conclusions

| Model | RMSE | MAE | R² |
|-------|------|-----|----|
| Naive baseline (persistence) | 0.4279 | 0.3266 | 0.7365 |
| CatBoost (Validation) | 0.2872 | 0.2114 | 0.8536 |
| CatBoost (Test) | 0.3398 | 0.2486 | 0.8338 |

- CatBoost reduces RMSE by ~21% over the naive baseline on the held-out test set, explaining ~83% of variance in next-month early delinquency rates.
- **Lagged delinquency rates** are the dominant features, confirming strong month-to-month autocorrelation in delinquency series.
- **Disaster-related features** (lagged damage, event occurrence, rolling exposure) contribute meaningful additional signal on top of that baseline.
- The slight performance gap between validation and test suggests mild non-stationarity in the later period, but the model generalizes well overall.
- The monthly performance plot confirms the model tracks the aggregate trend closely, without systematic over- or under-prediction in any period.